# Whisper Transcription Server
1. Run all cells top to bottom
2. Copy the **Public URL** printed in the last cell
3. Set it as `COLAB_WHISPER_URL` in your local `.env`

In [ ]:
!pip install -q fastapi uvicorn pyngrok faster-whisper python-multipart nest-asyncio

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")  # https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import os, tempfile, nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from faster_whisper import WhisperModel
from pyngrok import ngrok

app = FastAPI()

SUPPORTED = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".webm", ".mp4"}

print("Loading model...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Ready.")

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...)):
    ext = os.path.splitext(file.filename)[-1].lower() if file.filename else ".wav"
    if ext not in SUPPORTED:
        raise HTTPException(400, f"Unsupported format '{ext}'. Supported: {sorted(SUPPORTED)}")
    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        tmp.write(await file.read())
        temp_path = tmp.name
    try:
        segments, info = model.transcribe(temp_path, beam_size=5, vad_filter=True, language="ml")
        text = " ".join(seg.text.strip() for seg in segments)
        return {"language": info.language, "text": text}
    finally:
        os.remove(temp_path)

nest_asyncio.apply()
ngrok.kill()
tunnel = ngrok.connect(8000)
print("=" * 50)
print("Public URL:", tunnel.public_url)
print("=" * 50)

uvicorn.run(app, host="0.0.0.0", port=8000)